# Análisis Pokémon — Modelo Relacional (SQLite)

**Backend:** `data/pokemon.db` · 9 tablas SQLite · 1,350 Pokémon · 9 generaciones · 18 tipos

Este notebook analiza los datos Pokémon usando un modelo **relacional**: los datos están distribuidos en tablas conectadas por claves foráneas y se consultan con SQL + pandas.

## ¿Cuándo usar datos relacionales?
- Preguntas analíticas que cruzan varias entidades (promedios, correlaciones, agrupaciones)
- Cuando un cambio en un dato debe reflejarse en todas las consultas automáticamente
- Estadísticas, exploración, entrenamiento de modelos de ML

## Prerrequisitos
```bash
python scripts/fetch.py      # Descarga datos de PokéAPI (solo la primera vez)
python scripts/build_db.py   # Construye data/pokemon.db
```

---
**Secciones:** Explorador de tablas | Tipos | Atributos físicos | Stats (BST) | Legendarios | Efectividad | Combinaciones | Consultas SQL | Visor de Pokémon

In [1]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from pathlib import Path

TYPE_COLORS = {
    'normal':'#A8A77A',   'fire':'#EE8130',    'water':'#6390F0',
    'electric':'#F7D02C', 'grass':'#7AC74C',   'ice':'#96D9D6',
    'fighting':'#C22E28', 'poison':'#A33EA1',  'ground':'#E2BF65',
    'flying':'#A98FF3',   'psychic':'#F95587', 'bug':'#A6B91A',
    'rock':'#B6A136',     'ghost':'#735797',   'dragon':'#6F35FC',
    'dark':'#705746',     'steel':'#B7B7CE',   'fairy':'#D685AD',
}

TYPE_ES = {
    'normal':'Normal',      'fire':'Fuego',     'water':'Agua',
    'electric':'Electrico', 'grass':'Planta',   'ice':'Hielo',
    'fighting':'Lucha',     'poison':'Veneno',  'ground':'Tierra',
    'flying':'Volador',     'psychic':'Psiquico','bug':'Bicho',
    'rock':'Roca',          'ghost':'Fantasma', 'dragon':'Dragon',
    'dark':'Siniestro',     'steel':'Acero',    'fairy':'Hada',
}

STAT_COLS   = ['hp', 'attack', 'defense', 'special-attack', 'special-defense', 'speed']
STAT_LABELS = ['HP', 'ATK', 'DEF', 'SP.ATK', 'SP.DEF', 'SPD']
GEN_MAP = {
    'generation-i':'Gen I',     'generation-ii':'Gen II',
    'generation-iii':'Gen III', 'generation-iv':'Gen IV',
    'generation-v':'Gen V',     'generation-vi':'Gen VI',
    'generation-vii':'Gen VII', 'generation-viii':'Gen VIII',
    'generation-ix':'Gen IX',
}
GEN_ORDER = ['Gen I','Gen II','Gen III','Gen IV','Gen V',
             'Gen VI','Gen VII','Gen VIII','Gen IX']

# Busca data/pokemon.db hacia arriba desde el CWD (funciona desde raíz o notebooks/)
DB = next((p / 'data' / 'pokemon.db'
           for p in [Path.cwd(), *Path.cwd().parents]
           if (p / 'data' / 'pokemon.db').exists()), None)
assert DB is not None, 'No se encontró data/pokemon.db. Ejecuta: python scripts/build_db.py'
con = sqlite3.connect(DB)

print('Setup listo. DB conectada.')


Setup listo. DB conectada.


## 0. Explorador de Tablas

Antes de analizar, revisemos qué contiene cada tabla: cuántas filas tiene, qué columnas expone y cómo se distribuyen los valores clave.

In [2]:
# Mapa del esquema: tablas, filas y columnas disponibles
tablas = [
    'pokemon','pokemon_stats','pokemon_types','pokemon_abilities',
    'pokemon_moves','species','types','abilities','moves'
]
resumen = []
for t in tablas:
    sample = pd.read_sql(f'SELECT * FROM {t} LIMIT 1', con)
    n      = pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', con).iloc[0, 0]
    resumen.append({'tabla': t, 'filas': n, 'columnas': len(sample.columns),
                    'campos': ', '.join(sample.columns)})
pd.DataFrame(resumen)

,tabla,filas,columnas,campos
0,pokemon,1350,6,"id, name, height, weight, base_experience, spr..."
1,pokemon_stats,8100,4,"pokemon_id, stat_name, base_value, effort"
2,pokemon_types,2115,3,"pokemon_id, type_name, slot"
3,pokemon_abilities,2926,4,"pokemon_id, ability_name, is_hidden, slot"
4,pokemon_moves,122446,4,"pokemon_id, move_name, learn_method, level_lea..."
5,species,1025,11,"id, name, color, shape, habitat, is_legendary,..."
6,types,21,7,"name, double_damage_to, half_damage_to, no_dam..."
7,abilities,371,2,"name, effect"
8,moves,937,6,"name, power, accuracy, pp, type_name, damage_c..."


In [3]:
# Muestra de las primeras filas de las tablas principales
from IPython.display import display
for t in ['pokemon', 'species', 'moves', 'abilities', 'types']:
    display(pd.read_sql(f'SELECT * FROM {t} LIMIT 5', con)
              .style.set_caption(t).set_table_styles(
                  [{'selector':'caption','props':[('color','#ffd700'),
                    ('font-size','13px'),('font-weight','bold')]}]))

,id,name,height,weight,base_experience,sprite_default
0,1,bulbasaur,7,69,64,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/1.png
1,2,ivysaur,10,130,142,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/2.png
2,3,venusaur,20,1000,236,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/3.png
3,4,charmander,6,85,62,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/4.png
4,5,charmeleon,11,190,142,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/5.png


,id,name,color,shape,habitat,is_legendary,is_mythical,generation,capture_rate,gender_rate,base_happiness
0,1,bulbasaur,green,quadruped,grassland,0,0,generation-i,45,1,70
1,2,ivysaur,green,quadruped,grassland,0,0,generation-i,45,1,70
2,3,venusaur,green,quadruped,grassland,0,0,generation-i,45,1,70
3,4,charmander,red,upright,mountain,0,0,generation-i,45,1,70
4,5,charmeleon,red,upright,mountain,0,0,generation-i,45,1,70


,name,power,accuracy,pp,type_name,damage_class
0,10-000-000-volt-thunderbolt,195.000000,nan,1,electric,special
1,absorb,20.000000,100.000000,25,grass,special
2,accelerock,40.000000,100.000000,20,rock,physical
3,acid-armor,nan,nan,20,poison,status
4,acid-downpour--physical,nan,nan,1,poison,physical


,name,effect
0,adaptability,"This Pokémon inflicts twice as much damage with moves whose types match its own, rather than the usual same-type attack bonus of 1.5×."
1,aerilate,Turns the bearer's Normal-type moves into Flying-type moves. Moves changed by this ability have 1.3× their power.
2,aftermath,"When this Pokémon is knocked out by a move that makes contact, the move's user takes 1/4 its maximum HP in damage."
3,air-lock,"While this Pokémon is in battle, weather can still be in play, but will not have any of its effects. This ability functions identically to Cloud Nine."
4,analytic,This Pokémon's moves have 1.3× their power when it moves last in a turn. Future Sight and Doom Desire are unaffected.


,name,double_damage_to,half_damage_to,no_damage_to,double_damage_from,half_damage_from,no_damage_from
0,bug,"grass,psychic,dark","fighting,flying,poison,ghost,steel,fire,fairy",,"flying,rock,fire","fighting,ground,grass",
1,dark,"ghost,psychic","fighting,dark,fairy",,"fighting,bug,fairy","ghost,dark",psychic
2,dragon,dragon,steel,fairy,"ice,dragon,fairy","fire,water,grass,electric",
3,electric,"flying,water","grass,electric,dragon",ground,ground,"flying,steel,electric",
4,fairy,"fighting,dragon,dark","poison,steel,fire",,"poison,steel","fighting,bug,dark",dragon


In [4]:
# Estadísticas descriptivas de columnas numéricas clave (pokemon + species)
df_explore = pd.read_sql('''
    SELECT p.height, p.weight, p.base_experience,
           s.capture_rate, s.base_happiness, s.gender_rate
    FROM pokemon p LEFT JOIN species s ON p.id = s.id
''', con)
df_explore.describe().round(1)

,height,weight,base_experience,capture_rate,base_happiness,gender_rate
count,1350.0,1350.0,1302.0,1025.0,1025.0,1025.0
mean,20.5,991.7,161.9,95.5,59.9,2.9
std,53.3,1953.7,82.5,76.0,22.5,2.2
min,1.0,0.0,36.0,3.0,0.0,-1.0
25%,6.0,94.2,71.0,45.0,50.0,1.0
50%,11.0,327.0,165.0,60.0,70.0,4.0
75%,17.0,854.2,222.5,140.0,70.0,4.0
max,1000.0,10000.0,608.0,255.0,140.0,8.0


In [5]:
# Valores nulos por columna — cuántos registros faltan y qué porcentaje representan
nulos = df_explore.isnull().sum().rename('nulos')
pct   = (df_explore.isnull().mean() * 100).round(1).rename('%')
pd.concat([nulos, pct], axis=1)

,nulos,%
height,0,0.0
weight,0,0.0
base_experience,48,3.6
capture_rate,325,24.1
base_happiness,325,24.1
gender_rate,325,24.1


In [6]:
# Distribución de valores categóricos clave
from IPython.display import display
for nombre, tabla, campo in [
    ('Generaciones',   'species',       'generation'),
    ('Colores',        'species',       'color'),
    ('Tipos únicos',   'pokemon_types', 'type_name'),
    ('Clases de daño', 'moves',         'damage_class'),
]:
    n = pd.read_sql(
        f'SELECT COUNT(DISTINCT {campo}) as n FROM {tabla}', con).iloc[0, 0]
    vals = pd.read_sql(
        f'SELECT {campo}, COUNT(*) as cnt FROM {tabla} '
        f'GROUP BY {campo} ORDER BY cnt DESC LIMIT 5', con)
    print(f'\n{nombre}  ({n} valores únicos):')
    display(vals)


Generaciones  (9 valores únicos):


,generation,cnt
0,generation-v,156
1,generation-i,151
2,generation-iii,135
3,generation-ix,120
4,generation-iv,107



Colores  (10 valores únicos):


,color,cnt
0,blue,170
1,brown,147
2,green,130
3,gray,107
4,red,97



Tipos únicos  (18 valores únicos):


,type_name,cnt
0,water,192
1,normal,160
2,grass,156
3,flying,154
4,psychic,141



Clases de daño  (3 valores únicos):


,damage_class,cnt
0,physical,395
1,status,277
2,special,265


## 1. Carga y Vista General del Dataset

> **Nota — formas base vs formas alternativas:**  
> La BD tiene 1 350 entradas en la tabla `pokemon`, pero **325 son formas alternativas**
> (mega, regional, gmax, formas de batalla: `deoxys-attack`, `giratina-origin`…).
> Estas formas usan `id ≥ 10 000` en PokéAPI. Los análisis de las secciones siguientes
> filtran a las **1 025 formas base** (`id < 10 000`) para que las estadísticas reflejen
> el Pokédex nacional real. Las formas siguen accesibles en el Visor (sección 9).


In [7]:
# Formas alternativas (mega, regional, gmax…) usan id >= 10000 en PokéAPI.
# Se filtran para que los análisis reflejen las 1025 formas base del Pokédex nacional.
pokemon  = pd.read_sql('SELECT * FROM pokemon WHERE id < 10000', con)
stats_l  = pd.read_sql('SELECT * FROM pokemon_stats', con)
species  = pd.read_sql('SELECT * FROM species', con)
tipos_db = pd.read_sql('SELECT * FROM pokemon_types', con)

stats = (stats_l
         .pivot(index='pokemon_id', columns='stat_name', values='base_value')
         .reset_index())
stats.columns.name = None

tipo1 = (tipos_db[tipos_db['slot']==1][['pokemon_id','type_name']]
         .rename(columns={'type_name':'type1'}))
tipo2 = (tipos_db[tipos_db['slot']==2][['pokemon_id','type_name']]
         .rename(columns={'type_name':'type2'}))

df = (pokemon
      .merge(stats, left_on='id', right_on='pokemon_id', how='left').drop(columns='pokemon_id')
      .merge(tipo1, left_on='id', right_on='pokemon_id', how='left').drop(columns='pokemon_id')
      .merge(tipo2, left_on='id', right_on='pokemon_id', how='left').drop(columns='pokemon_id')
      .merge(species[['id','is_legendary','is_mythical','generation',
                       'capture_rate','base_happiness']], on='id', how='left'))

df['bst']       = df[STAT_COLS].sum(axis=1)
df['categoria'] = 'Normal'
df.loc[df['is_legendary'] == 1, 'categoria'] = 'Legendario'
df.loc[df['is_mythical']  == 1, 'categoria'] = 'Mitico'
df['dual_tipo'] = df['type2'].notna()
df['height_m']  = df['height'] / 10
df['weight_kg'] = df['weight'] / 10
df['gen']       = df['generation'].map(GEN_MAP).fillna(df['generation'])

n_pok  = len(df)
n_leg  = (df['categoria']=='Legendario').sum()
n_mit  = (df['categoria']=='Mitico').sum()
n_dual = df['dual_tipo'].sum()

print('=' * 50)
print('  Total Pokemon    :', n_pok, '(formas base — excluye 325 formas alt.)')
print('  Generaciones     :', df['gen'].nunique())
print('  Legendarios      :', n_leg, ' | Miticos:', n_mit)
print('  Dual-tipo        :', n_dual, '(%s%%)' % round(n_dual/n_pok*100, 1))
print('  BST promedio     :', round(df['bst'].mean(), 1),
      ' | Mediana:', df['bst'].median())
print('=' * 50)
df[['id','name','type1','type2','bst','gen','categoria']].head(8)


  Total Pokemon    : 1025 (formas base — excluye 325 formas alt.)
  Generaciones     : 9
  Legendarios      : 71  | Miticos: 23
  Dual-tipo        : 526 (51.3%)
  BST promedio     : 427.7  | Mediana: 450.0


,id,name,type1,type2,bst,gen,categoria
0,1,bulbasaur,grass,poison,318,Gen I,Normal
1,2,ivysaur,grass,poison,405,Gen I,Normal
2,3,venusaur,grass,poison,525,Gen I,Normal
3,4,charmander,fire,NaN,309,Gen I,Normal
4,5,charmeleon,fire,NaN,405,Gen I,Normal
5,6,charizard,fire,flying,534,Gen I,Normal
6,7,squirtle,water,NaN,314,Gen I,Normal
7,8,wartortle,water,NaN,405,Gen I,Normal


## 2. Distribución de Tipos Primarios

¿Cuántos Pokémon tienen cada tipo como tipo primario? El color de cada barra
corresponde al color canónico del tipo. Pasa el cursor para ver el valor exacto.


In [8]:
MAIN = sorted(TYPE_COLORS.keys())
type_counts = (df[df['type1'].isin(MAIN)]['type1']
               .value_counts()
               .reset_index()
               .rename(columns={'type1':'tipo', 'count':'n'})
               .sort_values('n'))

media = type_counts['n'].mean()

fig = px.bar(
    type_counts, x='n', y='tipo', orientation='h',
    color='tipo', color_discrete_map=TYPE_COLORS,
    text='n',
    labels={'n': 'Pokémon', 'tipo': 'Tipo primario'},
    title='Distribución de Tipos Primarios<br><sup>Número de Pokémon con cada tipo como tipo principal</sup>',
    template='plotly_dark',
)
fig.update_traces(textposition='outside')
fig.add_vline(x=media, line_dash='dash', line_color='#ffd700',
              annotation_text=f'Media: {media:.0f}',
              annotation_position='top right')
fig.update_layout(showlegend=False, height=580,
                  xaxis_title='Pokémon con ese tipo como tipo primario')
fig.show()


## 3. Atributos Físicos — Peso y Altura

**Scatter** (primer gráfico): cada punto es un Pokémon — posición horizontal = peso,
posición vertical = altura, **color = tipo primario**, tamaño proporcional al BST.
Pasa el cursor para ver nombre, tipo secundario, BST y medidas exactas.
Las líneas punteadas marcan las medianas de peso y altura.

**Top 10** (segundo y tercer gráfico): los récords absolutos del Pokédex.


In [9]:
ds = df[df['type1'].isin(MAIN) & df['weight_kg'].notna() & df['height_m'].notna()].copy()
ds = ds[(ds['weight_kg'] <= ds['weight_kg'].quantile(0.99)) &
        (ds['height_m']  <= ds['height_m'].quantile(0.99))]

fig = px.scatter(
    ds, x='weight_kg', y='height_m',
    color='type1', color_discrete_map=TYPE_COLORS,
    size='bst', size_max=18,
    hover_name='name',
    hover_data={'type1': True, 'type2': True, 'bst': True,
                'weight_kg': ':.1f', 'height_m': ':.1f'},
    labels={'weight_kg': 'Peso (kg)', 'height_m': 'Altura (m)',
            'type1': 'Tipo primario', 'bst': 'BST'},
    title='Peso vs Altura — coloreado por tipo primario<br>'
          '<sup>Tamaño proporcional al BST | Excluye top 1% de outliers | Pasa el cursor para ver el nombre</sup>',
    template='plotly_dark',
)
fig.add_hline(y=ds['height_m'].median(), line_dash='dot', line_color='#aaa',
              annotation_text=f"Med. altura: {ds['height_m'].median():.1f} m",
              annotation_position='top right')
fig.add_vline(x=ds['weight_kg'].median(), line_dash='dot', line_color='#aaa',
              annotation_text=f"Med. peso: {ds['weight_kg'].median():.0f} kg",
              annotation_position='top right')
fig.update_layout(legend_title_text='Tipo primario', height=580)
fig.show()


In [10]:
for col, unit, title in [
    ('weight_kg', 'kg', 'Top 10 más pesados'),
    ('height_m',  'm',  'Top 10 más altos'),
]:
    top = df.nlargest(10, col)[['name', col]].sort_values(col)
    fig = px.bar(
        top, x=col, y='name', orientation='h',
        color=col,
        color_continuous_scale='Blues' if unit == 'kg' else 'Reds',
        labels={col: f'({unit})', 'name': ''},
        title=title,
        template='plotly_dark',
        text=top[col].apply(lambda v: f'{v:.1f} {unit}'),
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(coloraxis_showscale=False, height=420)
    fig.show()


## 4. Estadísticas Base (BST)

**BST** (*Base Stat Total*) = suma de los 6 stats base de un Pokémon:
HP, Ataque, Defensa, Ataque Especial, Defensa Especial y Velocidad.
Un BST alto no implica un Pokémon bueno en todo — puede concentrarse en un solo stat.

- **Heatmap**: promedio de cada stat por tipo primario + columna `Prom.` (BST ÷ 6).
  El color indica la posición relativa del tipo en ese stat (amarillo = mínimo, rojo = máximo).
  El número en cada celda es el promedio real. Pasa el cursor para ver el valor exacto.
- **Boxplot**: distribución completa del BST por generación — mediana, cuartiles y outliers.
- **Tendencia**: evolución del BST medio generación a generación.


In [11]:
by_type = df[df['type1'].isin(MAIN)].groupby('type1')[STAT_COLS].mean()
by_type['Prom.'] = by_type[STAT_COLS].mean(axis=1)   # BST ÷ 6 por tipo

all_stat_cols   = STAT_COLS + ['Prom.']
all_stat_labels = STAT_LABELS + ['Prom.']

normed = (by_type[all_stat_cols] - by_type[all_stat_cols].min()) \
       / (by_type[all_stat_cols].max() - by_type[all_stat_cols].min())
normed = normed.sort_values('hp', ascending=False)
raw    = by_type.loc[normed.index][all_stat_cols].round(1)

fig = go.Figure(data=go.Heatmap(
    z=normed.values,
    x=all_stat_labels,
    y=[TYPE_ES.get(t, t) for t in normed.index],
    colorscale='YlOrRd',
    zmin=0, zmax=1,
    text=raw.values.astype(int),
    texttemplate='%{text}',
    hovertemplate='Tipo: %{y}<br>Stat: %{x}<br>Promedio: %{text}<extra></extra>',
    colorbar=dict(title='Posición<br>relativa'),
))
# Línea divisoria entre stats individuales y columna Prom.
fig.add_shape(
    type='line', x0=5.5, x1=5.5,
    y0=-0.5, y1=len(normed) - 0.5,
    xref='x', yref='y',
    line=dict(color='#aaa', width=2, dash='dot'),
)
fig.update_layout(
    title='Stats base promedio por tipo primario<br>'
          '<sup>Color = posición relativa en el stat | Número = promedio real | '
          'Prom. = BST ÷ 6</sup>',
    template='plotly_dark',
    height=600,
    xaxis_title='Estadística',
    yaxis_title='Tipo primario',
)
fig.show()


In [12]:
df_gen = df[df['gen'].isin(GEN_ORDER)].copy()

fig_box = px.box(
    df_gen, x='gen', y='bst', color='gen',
    category_orders={'gen': GEN_ORDER},
    labels={'gen': 'Generación', 'bst': 'BST'},
    title='Distribución de BST por Generación<br><sup>Pasa el cursor para ver mediana, cuartiles y outliers</sup>',
    template='plotly_dark',
)
fig_box.update_layout(showlegend=False, height=480)
fig_box.show()

sg = (df_gen.groupby('gen')['bst']
      .agg(media='mean', mediana='median', n='count')
      .reindex(GEN_ORDER).dropna())

fig_trend = px.line(
    sg.reset_index(), x='gen', y='media', markers=True,
    labels={'gen': 'Generación', 'media': 'BST medio'},
    title='Evolución del BST promedio por Generación',
    template='plotly_dark',
)
fig_trend.add_scatter(
    x=sg.index, y=sg['mediana'],
    mode='lines+markers', name='Mediana BST',
    line=dict(dash='dash', color='#87ceeb'),
)
for gen, row in sg.iterrows():
    fig_trend.add_annotation(
        x=gen, y=row['media'],
        text=f'n={int(row["n"])}',
        showarrow=False, yshift=14,
        font=dict(size=9, color='#aaa'),
    )
fig_trend.update_layout(height=450)
fig_trend.show()


In [13]:
cols_c = STAT_COLS + ['bst', 'height_m', 'weight_kg']
labs_c = STAT_LABELS + ['BST', 'Altura (m)', 'Peso (kg)']
corr   = df[cols_c].corr()
corr.index   = labs_c
corr.columns = labs_c

fig = px.imshow(
    corr,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto='.2f',
    title='Matriz de Correlación — Stats base, BST y atributos físicos<br>'
          '<sup>Azul = correlación positiva | Rojo = correlación negativa</sup>',
    template='plotly_dark',
    aspect='equal',
)
fig.update_layout(height=540,
                  coloraxis_colorbar=dict(title='Pearson'),
                  xaxis_tickangle=45)
fig.show()


## 5. Legendarios, Míticos y Normales

**Violines** agrupados por stat. El ancho del violín indica cuántos Pokémon tienen ese
valor exacto (más ancho = más frecuente). La caja interna muestra mediana y cuartiles.
Pasa el cursor para ver los valores precisos.

¿Los legendarios destacan en todos los stats por igual, o concentran el poder en alguno en particular?


In [14]:
cat_colors = {'Normal': '#4a9eff', 'Legendario': '#ffd700', 'Mitico': '#ff6b9d'}

stat_long = (df[['categoria'] + STAT_COLS]
             .melt(id_vars='categoria', var_name='stat', value_name='valor'))
stat_long['stat'] = stat_long['stat'].map(dict(zip(STAT_COLS, STAT_LABELS)))

fig = px.violin(
    stat_long, x='stat', y='valor',
    color='categoria', color_discrete_map=cat_colors,
    box=True, points=False,
    category_orders={'stat': STAT_LABELS,
                     'categoria': ['Normal', 'Legendario', 'Mitico']},
    labels={'stat': 'Estadística', 'valor': 'Valor', 'categoria': 'Categoría'},
    title='Distribución de Stats por Categoría<br>'
          '<sup>Normal vs Legendario vs Mítico | Pasa el cursor para ver mediana y cuartiles</sup>',
    template='plotly_dark',
)
fig.update_layout(violinmode='group', height=540, legend_title_text='Categoría')
fig.show()


## 6. Efectividad de Tipos

Multiplicadores de daño cuando el **tipo atacante** (fila) ataca al **tipo defensor** (columna):

| Símbolo | Multiplicador | Significado |
|---------|:---:|---|
| `2×` | ×2 | Super eficaz — el atacante tiene ventaja |
| `·` | ×1 | Normal — sin ventaja ni desventaja |
| `½×` | ×0.5 | Poco eficaz — el defensor resiste |
| `0×` | ×0 | Sin efecto — inmunidad total |

Las etiquetas de los tipos van **coloreadas** con el color canónico de cada tipo.
Pasa el cursor para ver atacante, defensor y multiplicador exacto.


In [15]:
types_df  = pd.read_sql('SELECT * FROM types', con)
all_types = sorted(t for t in types_df['name'].tolist() if t in TYPE_COLORS)
matrix    = pd.DataFrame(1.0, index=all_types, columns=all_types)

for _, row in types_df.iterrows():
    atk = row['name']
    if atk not in matrix.index: continue
    for col_field, val in [('double_damage_to', 2.0),
                            ('half_damage_to',   0.5),
                            ('no_damage_to',     0.0)]:
        for t in (row[col_field] or '').split(','):
            t = t.strip()
            if t in matrix.columns:
                matrix.loc[atk, t] = val

ANNOT_EFF = {0.0: '0×', 0.5: '½×', 1.0: '·', 2.0: '2×'}
text_eff  = [[ANNOT_EFF[matrix.loc[a, d]] for d in all_types] for a in all_types]
n_t       = len(all_types)

fig = go.Figure(data=go.Heatmap(
    z=matrix.values,
    x=[TYPE_ES.get(t, t) for t in all_types],
    y=[TYPE_ES.get(t, t) for t in all_types],
    colorscale=[[0,'#111111'],[0.25,'#c0392b'],[0.5,'#ecf0f1'],[1,'#27ae60']],
    zmin=0, zmax=2,
    text=text_eff,
    texttemplate='%{text}',
    hovertemplate='Atacante: %{y}<br>Defensor: %{x}<br>Multiplicador: %{z}×<extra></extra>',
    colorbar=dict(
        title='Multiplicador',
        tickvals=[0, 0.5, 1, 2],
        ticktext=['0× sin efecto', '½× poco eficaz', '1× normal', '2× super eficaz'],
    ),
))

# Etiquetas coloreadas — ocultar tick labels por defecto y añadir anotaciones
for t in all_types:
    t_es = TYPE_ES.get(t, t)
    clr  = TYPE_COLORS[t]
    # Eje Y (tipo atacante, izquierda)
    fig.add_annotation(
        x=-0.01, xref='paper', xanchor='right',
        y=t_es, yref='y',
        text=f'<b>{t_es}</b>',
        showarrow=False,
        font=dict(color=clr, size=11),
    )
    # Eje X (tipo defensor, abajo)
    fig.add_annotation(
        x=t_es, xref='x',
        y=-0.01, yref='paper', yanchor='top',
        text=f'<b>{t_es}</b>',
        showarrow=False,
        textangle=-45,
        font=dict(color=clr, size=11),
    )

fig.update_layout(
    title='Matriz de Efectividad de Tipos<br>'
          '<sup>Fila = tipo atacante | Columna = tipo defensor | '
          'Etiquetas coloreadas con el color canónico del tipo</sup>',
    template='plotly_dark',
    height=800,
    xaxis=dict(showticklabels=False, showgrid=False, title='Tipo defensor'),
    yaxis=dict(showticklabels=False, showgrid=False,
               autorange='reversed', title='Tipo atacante'),
    margin=dict(l=120, b=160, r=20, t=100),
)
fig.show()


## 7. Combinaciones de Tipos

**¿Qué tipos aparecen juntos y con qué frecuencia?** Solo pares dual-tipo (Pokémon con dos tipos distintos); los mono-tipo ya están en la Sección 2.

- **Ranking** (primera gráfica): las ~15 combinaciones más comunes, ordenadas de mayor a menor. Responde de un vistazo: *¿cuáles pares existen en más Pokémon?*
- **Heatmap** (segunda gráfica): panorama completo del espacio de 18×18 tipos. Cada celda del triángulo superior es un par dual — el color y el número muestran cuántos Pokémon lo tienen. Las celdas vacías (blanco) son pares que no existen en el Pokédex. Pasa el cursor para ver el conteo exacto.


In [16]:
from collections import Counter
import numpy as np

tipo1_df = pd.read_sql(
    'SELECT pokemon_id, type_name FROM pokemon_types WHERE slot=1 AND pokemon_id < 10000', con)
tipo2_df = pd.read_sql(
    'SELECT pokemon_id, type_name FROM pokemon_types WHERE slot=2 AND pokemon_id < 10000', con)
comb = tipo1_df.merge(tipo2_df, on='pokemon_id', how='left', suffixes=('_1','_2'))

# Contar cada par dual como conjunto no ordenado
pair_counts = Counter()
for _, row in comb.iterrows():
    t1, t2 = row['type_name_1'], row['type_name_2']
    if pd.isna(t2): continue
    if t1 in TYPE_COLORS and t2 in TYPE_COLORS:
        pair_counts[tuple(sorted((t1, t2)))] += 1

top_n   = 15
top     = pair_counts.most_common(top_n)
labels  = [f"{TYPE_ES.get(a,a)} + {TYPE_ES.get(b,b)}" for (a,b),_ in top]
values  = [c for _,c in top]

# Ordenar de menor a mayor para que la mayor quede arriba en barras horizontales
order   = sorted(range(top_n), key=lambda i: values[i])
labels  = [labels[i] for i in order]
values  = [values[i] for i in order]

fig_bar = px.bar(
    x=values, y=labels,
    orientation='h',
    color=values,
    color_continuous_scale='Teal',
    labels={'x': 'Pokémon', 'y': 'Combinación', 'color': 'Pokémon'},
    title=f'Top {top_n} combinaciones dual-tipo más comunes',
    template='plotly_dark',
    text=values,
)
fig_bar.update_traces(textposition='outside')
fig_bar.update_layout(
    height=500,
    coloraxis_showscale=False,
    yaxis=dict(tickfont=dict(size=12)),
    margin=dict(l=160, r=60),
)
fig_bar.show()


In [17]:
# Heatmap simétrico — solo pares dual (diagonal excluida para que la escala sea útil)
all_t = sorted(TYPE_COLORS.keys())
n_ty  = len(all_t)

sym = pd.DataFrame(0, index=all_t, columns=all_t, dtype=int)
for _, row in comb.iterrows():
    t1, t2 = row['type_name_1'], row['type_name_2']
    if pd.isna(t2): continue
    if t1 in all_t and t2 in all_t:
        sym.loc[t1, t2] += 1
        sym.loc[t2, t1] += 1

sym_z = sym.values.copy().astype(float)
sym_z[np.tril_indices(n_ty, k=-1)] = float('nan')  # triángulo inferior → NaN
np.fill_diagonal(sym_z, float('nan'))               # diagonal mono-tipo → NaN

lbl = [TYPE_ES.get(t, t) for t in all_t]
text_h = [['' if np.isnan(sym_z[i, j]) else (str(int(sym_z[i, j])) if sym_z[i, j] > 0 else '')
           for j in range(n_ty)] for i in range(n_ty)]

fig_heat = go.Figure(data=go.Heatmap(
    z=sym_z,
    x=lbl, y=lbl,
    colorscale='YlGnBu',
    text=text_h,
    texttemplate='%{text}',
    hovertemplate='%{y} + %{x}: %{text} Pokémon<extra></extra>',
    colorbar=dict(title='Pokémon'),
))
fig_heat.update_layout(
    title=(
        'Matriz de combinaciones dual-tipo — triángulo superior<br>'
        '<sup>Celdas vacías = combinación inexistente en el Pokédex nacional</sup>'
    ),
    template='plotly_dark',
    height=680,
    xaxis=dict(title='Tipo B', tickangle=45, side='bottom'),
    yaxis=dict(title='Tipo A', autorange='reversed'),
)
fig_heat.show()


## 8. Consultas SQL Personalizadas

In [18]:
pd.read_sql('''
    SELECT name, type_name AS tipo, power AS poder,
           accuracy AS precision, pp, damage_class AS clase
    FROM moves
    WHERE power IS NOT NULL
    ORDER BY power DESC
    LIMIT 15
''', con)


,name,tipo,poder,precision,pp,clase
0,explosion,normal,250,100.0,5,physical
1,catastropika,electric,210,NaN,1,physical
2,pulverizing-pancake,normal,210,NaN,1,physical
3,light-that-burns-the-sky,psychic,200,NaN,1,special
4,menacing-moonraze-maelstrom,ghost,200,NaN,1,special
5,searing-sunraze-smash,steel,200,NaN,1,physical
6,self-destruct,normal,200,100.0,5,physical
7,10-000-000-volt-thunderbolt,electric,195,NaN,1,special
8,oceanic-operetta,water,195,NaN,1,special
9,soul-stealing-7-star-strike,ghost,195,NaN,1,physical


In [19]:
pd.read_sql('''
    SELECT pt.type_name AS tipo, sp.generation AS generacion,
           ROUND(AVG(sp.capture_rate), 1) AS captura_promedio,
           COUNT(*) AS n_pokemon
    FROM pokemon_types pt
    JOIN species sp ON pt.pokemon_id = sp.id
    WHERE pt.slot = 1 AND sp.capture_rate IS NOT NULL
    GROUP BY pt.type_name, sp.generation
    ORDER BY captura_promedio ASC
    LIMIT 20
''', con)


,tipo,generacion,captura_promedio,n_pokemon
0,flying,generation-v,3.0,1
1,steel,generation-vii,13.5,4
2,ice,generation-i,24.0,2
3,steel,generation-ii,25.0,2
4,dark,generation-iv,26.0,3
5,fairy,generation-iv,30.0,1
6,fighting,generation-ix,32.7,3
7,dragon,generation-iii,33.0,7
8,dragon,generation-v,33.4,7
9,fire,generation-iv,33.6,5


In [20]:
# Tabla de verificación: todos los Pokémon dual-tipo ordenados por ID
dual = pd.read_sql('''
    SELECT p.id, p.name,
           MAX(CASE WHEN pt.slot=1 THEN pt.type_name END) AS tipo1,
           MAX(CASE WHEN pt.slot=2 THEN pt.type_name END) AS tipo2
    FROM pokemon p
    JOIN pokemon_types pt ON p.id = pt.pokemon_id
    WHERE p.id < 10000
    GROUP BY p.id
    HAVING tipo2 IS NOT NULL
    ORDER BY p.id
''', con)

print(f"Total Pokémon con doble tipo: {len(dual)}")
dual.head(30)


Total Pokémon con doble tipo: 526


,id,name,tipo1,tipo2
0,1,bulbasaur,grass,poison
1,2,ivysaur,grass,poison
2,3,venusaur,grass,poison
3,6,charizard,fire,flying
4,12,butterfree,bug,flying
5,13,weedle,bug,poison
6,14,kakuna,bug,poison
7,15,beedrill,bug,poison
8,16,pidgey,normal,flying
9,17,pidgeotto,normal,flying


In [21]:
con.close()
print('Conexion cerrada.')

Conexion cerrada.
